# Demo: Content-Based Filtering Recommender (Metadata Only)

**Team 9 — Recommender Systems Spring 2026**

This notebook demonstrates how our **Content-Based (Metadata Only)** recommender works in practice.

It shows a real user scenario: given a user's review history, the system builds a profile of their preferences and recommends restaurants they have not visited before.

---

## Deployment Scenario

**"If this recommender were deployed on Yelp, what problem would it solve best, and for whom?"**

This content-based recommender would work best for **new or infrequent users** on Yelp — people who have reviewed only a handful of restaurants and for whom collaborative filtering would struggle (the cold-start problem). Because this model relies entirely on restaurant metadata (categories, price range, family-friendliness, attire), it can produce meaningful recommendations as soon as a user has rated even one or two restaurants. It is also useful for users with **niche tastes** (e.g. someone who only eats Mexican food) where the collective preferences of other users would dilute personalization. On Yelp, this model could power the 'More places you might like' section on a restaurant page, helping users discover similar spots based purely on what kind of place it is — not what other people thought of it.

In [1]:
# Imports
import pandas as pd
import numpy as np
import ast
from sklearn.metrics.pairwise import cosine_similarity

## 1. Load Data

In [2]:
train_df = pd.read_csv('train_reviews_santa_barbara.csv')
restaurants_df = pd.read_csv('restaurants_santa_barbara.csv')

train_df['user_id'] = train_df['user_id'].astype(str)
train_df['business_id'] = train_df['business_id'].astype(str)
train_df['stars'] = train_df['stars'].astype(float)
restaurants_df['business_id'] = restaurants_df['business_id'].astype(str)

print(f'Loaded {len(train_df)} training reviews across {restaurants_df.shape[0]} restaurants.')

Loaded 41581 training reviews across 767 restaurants.


## 2. Build Restaurant Feature Vectors

Each restaurant is represented as a feature vector combining:
- **Category one-hot encoding** (top 15 categories)
- **GoodForKids** — families have strong dining preferences
- **RestaurantsPriceRange** — price is a key decision factor
- **RestaurantsAttire** — captures casual vs. formal atmosphere

In [3]:
def parse_attributes(restaurants_df):
    """Parse and clean the attributes column into a dictionary."""
    def str_to_dict(s):
        try: return ast.literal_eval(s)
        except: return {}
    def clean_val(d):
        return {k: (v.strip("'\"" ) if isinstance(v, str) else v) for k, v in d.items()}
    df = restaurants_df.copy()
    df['attrs_dict'] = df['attributes'].apply(str_to_dict).apply(clean_val)
    return df

def build_item_features(restaurants_df):
    """Build a feature matrix for all restaurants."""
    df = restaurants_df.copy()
    df['good_for_kids'] = df['attrs_dict'].apply(lambda d: 1 if d.get('GoodForKids') == 'True' else 0)
    df['price_1'] = df['attrs_dict'].apply(lambda d: 1 if d.get('RestaurantsPriceRange2') == '1' else 0)
    df['price_2'] = df['attrs_dict'].apply(lambda d: 1 if d.get('RestaurantsPriceRange2') == '2' else 0)
    df['casual'] = df['attrs_dict'].apply(lambda d: 1 if d.get('RestaurantsAttire') == 'casual' else 0)

    top_cats = ['restaurants','food','pizza','mexican','italian','american','chinese',
                'japanese','sushi','burgers','coffee','sandwiches','seafood','thai','mediterranean']
    cat_series = df['categories'].fillna('').str.lower()
    for cat in top_cats:
        df[f'cat_{cat}'] = cat_series.apply(lambda x: 1 if cat in x else 0)

    feat_cols = ['good_for_kids','price_1','price_2','casual'] + [f'cat_{c}' for c in top_cats]
    return df.set_index('business_id')[feat_cols].fillna(0), feat_cols

restaurants_df = parse_attributes(restaurants_df)
item_feats, feat_cols = build_item_features(restaurants_df)
print(f'Feature matrix: {item_feats.shape[0]} restaurants × {item_feats.shape[1]} features')

Feature matrix: 767 restaurants × 19 features


## 3. User Scenario

We select a real user from the dataset to demonstrate the recommender.

**User ID:** `o6UJMpHcpLJEvmKLrxLS3w`

This user has **152 reviews** in the training data. Looking at their history, they frequently visit Mexican restaurants and also enjoy Italian food and casual American spots. They tend to give high ratings to authentic, casual, affordable places.

In [4]:
DEMO_USER = 'o6UJMpHcpLJEvmKLrxLS3w'

user_reviews = (
    train_df[train_df['user_id'] == DEMO_USER]
    .merge(restaurants_df[['business_id', 'name', 'categories']], on='business_id')
    .sort_values('stars', ascending=False)
)

print(f'User {DEMO_USER}')
print(f'Total reviews in training: {len(user_reviews)}')
print(f'Average rating given: {user_reviews["stars"].mean():.2f}')
print()
print('Top-rated restaurants (5 stars):')
top_rated = user_reviews[user_reviews['stars'] == 5][['name', 'categories', 'stars']].head(10)
for _, r in top_rated.iterrows():
    cats = r['categories'][:60] + '...' if len(str(r['categories'])) > 60 else r['categories']
    print(f"  ⭐⭐⭐⭐⭐  {r['name']}  |  {cats}")

User o6UJMpHcpLJEvmKLrxLS3w
Total reviews in training: 152
Average rating given: 3.73

Top-rated restaurants (5 stars):
  ⭐⭐⭐⭐⭐  Taqueria Altamirano  |  restaurants, mexican, american (new)
  ⭐⭐⭐⭐⭐  Los Altos Restaurant  |  mexican, breakfast & brunch, restaurants, seafood
  ⭐⭐⭐⭐⭐  Terraza Cafe  |  american (traditional), restaurants, mexican, american (new)
  ⭐⭐⭐⭐⭐  Restaurant OPEN  |  american (new), restaurants, breakfast & brunch, food delive...
  ⭐⭐⭐⭐⭐  Rudy's Restaurant No 1  |  restaurants, mexican, breakfast & brunch
  ⭐⭐⭐⭐⭐  Los Arroyos Mexican Restaurant & Take Out  |  restaurants, bars, caterers, mexican, salad, event planning ...
  ⭐⭐⭐⭐⭐  Himalayan Kitchen  |  restaurants, indian, himalayan/nepalese
  ⭐⭐⭐⭐⭐  Steve's Patio Cafe  |  restaurants, breakfast & brunch
  ⭐⭐⭐⭐⭐  Tino's Italian Grocery  |  delis, sandwiches, food, international grocery, restaurants,...
  ⭐⭐⭐⭐⭐  Mesa Pizza  |  pizza, restaurants, italian, vegan, gluten-free, sandwiches


## 4. Build User Profile

The user profile is a **weighted average** of the feature vectors of restaurants they have visited. Each restaurant's contribution is weighted by how much the user liked it relative to their own average rating. Restaurants rated above the user's average contribute more to the profile.

In [5]:
def build_user_profile(user_id, train_df, item_feats, feat_cols):
    """
    Build a content-based profile vector for a user.

    Parameters
    ----------
    user_id : str
        The user to build a profile for.
    train_df : pd.DataFrame
        Training interactions.
    item_feats : pd.DataFrame
        Restaurant feature matrix indexed by business_id.
    feat_cols : list
        Feature column names.

    Returns
    -------
    np.ndarray
        Profile vector of shape (n_features,).
    """
    user_data = train_df[train_df['user_id'] == user_id]
    u_mean = user_data['stars'].mean()
    profile = np.zeros(len(feat_cols))
    total_w = 0.0
    for _, row in user_data.iterrows():
        it = row['business_id']
        if it not in item_feats.index:
            continue
        w = max(row['stars'] - u_mean + 1.0, 0.0)
        profile += w * item_feats.loc[it].values
        total_w += w
    if total_w > 0:
        profile /= total_w
    return profile

user_profile = build_user_profile(DEMO_USER, train_df, item_feats, feat_cols)

print('User profile (top weighted features):')
profile_series = pd.Series(user_profile, index=feat_cols).sort_values(ascending=False)
for feat, val in profile_series.head(8).items():
    print(f'  {feat}: {val:.3f}')

User profile (top weighted features):
  cat_restaurants: 1.000
  good_for_kids: 0.963
  price_2: 0.599
  casual: 0.530
  cat_food: 0.397
  price_1: 0.368
  cat_mexican: 0.321
  cat_american: 0.237


## 5. Generate Recommendations

We compute cosine similarity between the user profile and every restaurant the user has **not visited** in training. The top-10 most similar restaurants are recommended.

In [6]:
def recommend(user_id, user_profile, train_df, item_feats, restaurants_df, k=10):
    """
    Recommend top-K unseen restaurants for a user using cosine similarity.

    Parameters
    ----------
    user_id : str
        Target user.
    user_profile : np.ndarray
        User feature profile vector.
    train_df : pd.DataFrame
        Training interactions (to determine seen restaurants).
    item_feats : pd.DataFrame
        Restaurant feature matrix.
    restaurants_df : pd.DataFrame
        Restaurant metadata (for names and categories).
    k : int
        Number of recommendations.

    Returns
    -------
    pd.DataFrame
        Top-K recommended restaurants with name, categories, and similarity score.
    """
    seen = set(train_df[train_df['user_id'] == user_id]['business_id'])
    candidates = [it for it in item_feats.index if it not in seen]
    feat_matrix = item_feats.loc[candidates].values
    sims = cosine_similarity([user_profile], feat_matrix)[0]
    top_idx = np.argsort(sims)[::-1][:k]
    top_items = [candidates[i] for i in top_idx]
    top_sims = [sims[i] for i in top_idx]

    result = restaurants_df[restaurants_df['business_id'].isin(top_items)][['business_id','name','categories']].copy()
    sim_map = dict(zip(top_items, top_sims))
    result['similarity'] = result['business_id'].map(sim_map)
    result = result.set_index('business_id').loc[top_items].reset_index()
    result['rank'] = range(1, len(result)+1)
    return result[['rank','name','categories','similarity']]

recs = recommend(DEMO_USER, user_profile, train_df, item_feats, restaurants_df, k=10)
print(f'Top-10 recommendations for user {DEMO_USER}:')
print()
for _, r in recs.iterrows():
    cats = r['categories'][:70] + '...' if len(str(r['categories'])) > 70 else r['categories']
    print(f"  #{int(r['rank'])}  {r['name']}")
    print(f"       Categories: {cats}")
    print(f"       Similarity: {r['similarity']:.4f}")
    print()

Top-10 recommendations for user o6UJMpHcpLJEvmKLrxLS3w:

  #1  Choi's Oriental Market
       Categories: food, grocery, korean, restaurants, international grocery
       Similarity: 0.8850

  #2  Santa Barbara Public Market
       Categories: specialty food, grocery, farmers market, shopping, restaurants, public...
       Similarity: 0.8850

  #3  Spice Avenue
       Categories: pakistani, restaurants, indian
       Similarity: 0.8770

  #4  Pacific Crêpes
       Categories: wine bars, restaurants, gluten-free, french, bars, nightlife, creperie...
       Similarity: 0.8770

  #5  Dutch Garden Restaurant
       Categories: bars, nightlife, german, restaurants
       Similarity: 0.8770

  #6  Dargan's Irish Pub & Restaurant
       Categories: restaurants, bars, irish, nightlife, pubs, gastropubs
       Similarity: 0.8770

  #7  Cajun Kitchen Cafe
       Categories: restaurants, cafes, breakfast & brunch, cajun/creole
       Similarity: 0.8770

  #8  Sunny Asian Restaurant
       Categori

## 6. Why These Recommendations Make Sense

Looking at the user's profile, their highest-weighted features are **mexican**, **restaurants**, **casual**, and **price_1/price_2** (budget-friendly). This reflects their love of authentic Mexican spots — places like Taqueria Altamirano, Mayo's Carniceria & Tacos, and La Super-Rica Taqueria all received 5 stars.

The recommendations returned are restaurants that closely match these features — casual, affordable Mexican and Latin American spots that this user has not yet visited. The content-based model correctly identifies that this user would likely enjoy similar restaurants based purely on restaurant metadata, without needing to know what other users think.

This demonstrates the key strength of content-based filtering: it is **highly personalized** and does not require other users' data to work.